# HAETAE 축 A — 하드웨어 클럭글리치 T2 (독립 노트북)

무방어 baseline(+대응기법 3종)에 **실제 클럭글리치**로 T2(+y 스킵)를 유발 → **결함 응답 z1의 s1 복원율**로 누설 판정.
`Lab_HAETAE_F4_FullSign_v1`(축 B / EXP1~6)과 **독립**. 이 노트북만 위→아래로 실행.

**핵심 설계(디버깅으로 확정):**
1. **트리거=공격지점**: 펌웨어 `T` 명령으로 트리거를 특정 연산(기본 ADDY=T2)에만 발생 → 윈도우 축소 → 정밀 타격. (리셋 시 초기화되므로 매 리셋 후 재전송)
2. **width sweet spot ≈ 50**(AES 검증에서 확정). width/offset 정수.
3. `glitch_once`는 `ext_single`로 글리치 자동발사 → **시리얼을 먼저 읽고 capture는 뒤에**(capture 타임아웃 간섭 방지).
4. **LEAK 판정 = s1 복원율(agreement) ≥ LEAK_TH**. HW 글리치는 명령어 1개만 교란해 LEAK_T2와 바이트일치가 거의 없으므로 exact 다이제스트로 판정하지 않음.

**순서:** EXP7-a(셋업) → EXP7-b2(width 밴드 진단) → EXP7-b(랜덤 탐색) → EXP7-c(정밀화) → EXP7-d(4변형 비교).
전제: BUILD된 `haetae-*-FSIM-CW308_STM32F4.hex`(트리거 펌웨어)가 펌웨어 폴더에 존재. ⚠ 서명 1회 ~8초.

In [ ]:
# ===== EXP7-a: 단독 부트스트랩 + '공격지점=트리거지점' 셋업 =====
%matplotlib inline
SCOPETYPE='OPENADC'; PLATFORM='CW308_STM32F4'; CRYPTO_TARGET='NONE'; SS_VER='SS_VER_1_1'
import chipwhisperer as cw
import numpy as np, time, struct, csv, collections, logging, random
logging.getLogger('ChipWhisperer').setLevel(logging.ERROR)
try:
    scope
except NameError:
    scope = cw.scope(name='Husky')
%run "../../Setup_Scripts/Setup_Generic.ipynb"

scope.clock.clkgen_freq = 7.37e6; scope.clock.adc_mul = 1; scope.io.hs2 = 'clkgen'
time.sleep(0.2)
import importlib, haetae_recover; importlib.reload(haetae_recover)
from haetae_recover import attack_recover

FW = '../../../firmware/mcu/simpleserial-haetae/'
FL = {'NONE':0,'SEED':1,'SIGNBIT':2,'UNPACK':3,'LSB':4,'CS':5,'ADDY':6,'REJECT':7}
GOLDEN  = bytes.fromhex('ba9f152c607b207fc6512635ba11388c')   # 무결함 서명(결정론)
LEAK_T2 = bytes.fromhex('63ff5ebfaa6263739651890939cccb48')   # T2 클린스킵 누설(참고용)
TRIG_POINT = FL['ADDY']       # ★ 공격지점=트리거지점. ADDY=T2(+y) / CS=T1 / REJECT=RB / 0=전체

def ss_echo(timeout=3000):
    target.flush(); target.simpleserial_write('e', bytearray())
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def ss_sign(timeout=90000):
    target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    r = target.simpleserial_read('r', 16, timeout=timeout); return bytes(r) if r else None
def set_fault(line, oneshot=0, checkskip=0):
    target.flush(); target.simpleserial_write('f', bytes([line, oneshot, checkskip]))
    return target.simpleserial_read('r', 1, timeout=3000)
def ss_trig(pt):
    target.flush(); target.simpleserial_write('T', bytes([pt]))
    return target.simpleserial_read('r', 1, timeout=3000)
def flash(hexname):
    cw.program_target(scope, prog, FW + hexname); reset_target(scope); time.sleep(0.5); target.flush()
def recover_target():                       # 리셋 후 트리거지점 재설정(리셋이 g_trig_line 초기화)
    reset_target(scope); time.sleep(0.5); target.flush()
    try: ss_trig(TRIG_POINT)
    except Exception: pass

# 플래시(clean clock) + SW결함 OFF + 워밍업(키생성) + GOLDEN 확인
scope.io.hs2 = 'clkgen'
flash('haetae-baseline-FSIM-{}.hex'.format(PLATFORM))
set_fault(FL['NONE'], 0, 0)
g = ss_sign()
print('echo:', (ss_echo() or b'').hex()[:8], '| sign:', g.hex() if g else None)
assert g == GOLDEN, 'GOLDEN 불일치! 펌웨어/키 확인'
pt_name = [k for k,v in FL.items() if v==TRIG_POINT][0]
print('trig set:', (ss_trig(TRIG_POINT) or b'').hex(), '( 지점 =', pt_name, ')')

# (1) 트리거 윈도우 측정 — TRIG_POINT 연산만 → 작게
scope.adc.basic_mode='rising_edge'
try: scope.trigger.triggers='tio4'
except Exception: pass
scope.adc.timeout = 12
scope.arm(); target.simpleserial_write('p', bytearray([0]*16))
to = scope.capture(); WIN = int(scope.adc.trig_count)
target.simpleserial_read('r', 16, timeout=20000)
print('WIN(%s만) = %d 타겟 사이클 | timeout? %s' % (pt_name, WIN, to))

# (2) 클럭글리치 모드 + 리셋 + 트리거지점 재설정
scope.glitch.enabled = True
scope.glitch.clk_src = 'pll'; scope.glitch.output = 'clock_xor'
scope.glitch.trigger_src = 'ext_single'; scope.glitch.repeat = 1
scope.io.hs2 = 'glitch'; scope.adc.timeout = 3
time.sleep(0.2); reset_target(scope); time.sleep(0.5); target.flush(); ss_trig(TRIG_POINT)
PSS = scope.glitch.phase_shift_steps
print('glitch 모드 | echo:', (ss_echo() or b'').hex()[:8], '| PSS =', PSS)

def set_glitch(delay, width, offset):
    scope.glitch.ext_offset = int(delay); scope.glitch.width = int(width); scope.glitch.offset = int(offset)

def glitch_once(read_timeout=18000):
    # ext_single → 글리치는 하드웨어 트리거에 자동 발사. 시리얼 먼저 읽고 capture는 뒤(정리용).
    scope.arm(); target.flush(); target.simpleserial_write('p', bytearray([0]*16))
    r = target.simpleserial_read_witherrors('r', 16, glitch_timeout=read_timeout)
    try: scope.capture()
    except Exception: pass
    if (not r['valid']) or r['payload'] is None:
        if ss_echo(800) is None: recover_target()
        else: target.flush()
        return 'mute', None
    p = bytes(r['payload'])
    return ('normal', p) if p == GOLDEN else (('success', p) if p == LEAK_T2 else ('other', p))

def recover_agreement():                    # 결함 응답 z1 → s1 복원율(0~1)
    try: return float(attack_recover(target)['agreement'])
    except Exception: return 0.0

print('준비 완료 — EXP7-b2(진단) 또는 EXP7-b(랜덤) 실행')

In [ ]:
# ===== EXP7-b2 (진단): TRIG_POINT에서 width 미세 스윕 — clean-fault 밴드 탐색 =====
mid = max(WIN // 2, 1)
cnt = collections.OrderedDict(golden=0, faulty=0, mute=0); rows=[]
print('width 0..70 @ %s (ext=%d, offset=0)' % ([k for k,v in FL.items() if v==TRIG_POINT][0], mid), flush=True)
for w in range(0, 71, 2):
    set_glitch(mid, w, 0)
    g, p = glitch_once()
    if g == 'mute': cls='mute'; agr=None
    elif g == 'normal': cls='golden'; agr=None
    else: agr = recover_agreement(); cls='faulty'
    cnt[cls] += 1; rows.append((w, cls, agr))
    print('w=%2d -> %-7s %s' % (w, cls, ('s1=%.0f%%'%(100*agr)) if agr is not None else ''), flush=True)
print('요약:', dict(cnt))
print('w=0은 golden이어야 정상. golden→faulty→mute 전이의 faulty(복원율 높은) 구간이 밴드.')
print('전부 mute(단, w=0도 mute면) → 상태문제/설정문제. w=0만 golden이고 이후 곧장 mute면 밴드가 좁은 것.')

In [ ]:
# ===== EXP7-b: 랜덤 글리치 탐색 — 복원율 기반 LEAK 판정 + tqdm 진행바 (그래프 끝 1회) =====
from tqdm.notebook import trange
N_TRIES  = 300
W_RANGE  = (35, 70); O_RANGE = (-15, 15); REP_POOL = [1, 1, 2]
E_MAX    = int(WIN * 1.05) if WIN > 0 else 4000
LEAK_TH  = 0.99
STOP_AFTER = 2
cnt = collections.OrderedDict(golden=0, LEAK=0, faulty=0, mute=0); log=[]; leakp=[]; best_agr=0.0
bar = trange(N_TRIES, desc='HAETAE glitch @%s' % [k for k,v in FL.items() if v==TRIG_POINT][0])
for i in bar:
    w = random.randint(*W_RANGE); o = random.randint(*O_RANGE); e = random.randint(0, E_MAX); rep = random.choice(REP_POOL)
    set_glitch(e, w, o); scope.glitch.repeat = int(rep)
    g, p = glitch_once(); agr = None
    if g == 'mute':      cls = 'mute'
    elif g == 'normal':  cls = 'golden'
    else:
        agr = recover_agreement(); best_agr = max(best_agr, agr)
        if agr >= LEAK_TH:
            cls = 'LEAK'; leakp.append((e, w, o, rep, round(agr,3)))
            bar.write('★ LEAK ext=%d w=%d o=%d rep=%d → s1 복원율 %.1f%%' % (e, w, o, rep, 100*agr))
        else: cls = 'faulty'
    cnt[cls] += 1
    log.append((int(e), int(w), int(o), int(rep), cls, ('%.3f'%agr) if agr is not None else '', p.hex() if p else ''))
    bar.set_postfix(best_s1='%.0f%%' % (100*best_agr), **cnt)
    if cnt['LEAK'] >= STOP_AFTER: bar.write('LEAK 확보 — 종료'); break
scope.glitch.repeat = 1
print('DONE:', dict(cnt), '| best s1 복원율 = %.1f%%' % (100*best_agr), '| leak:', leakp[:5])
with open('exp7_glitch_log.csv','w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['ext','width','offset','repeat','class','agreement','digest']); wr.writerows(log)
print('saved exp7_glitch_log.csv')

import matplotlib.pyplot as plt
CMAP = collections.OrderedDict(golden='0.8', LEAK='red', faulty='orange', mute='black')
fig, ax = plt.subplots(figsize=(9,3.6), dpi=120)
for cls,col in CMAP.items():
    pts=[(r[0],r[1]) for r in log if r[4]==cls]
    if pts:
        xs,ys=zip(*pts); ax.scatter(xs,ys,c=col,s=24,alpha=0.75,edgecolors='none',label='%s (%d)'%(cls,cnt[cls]))
ax.set_xlim(0,max(E_MAX,1)); ax.set_ylim(W_RANGE[0]-2,W_RANGE[1]+2)
ax.set_xlabel('ext_offset'); ax.set_ylabel('width'); ax.legend(loc='upper right',fontsize=8)
ax.set_title('HAETAE HW glitch (red=LEAK: s1 recovered, orange=faulty)')
fig.tight_layout(); fig.savefig('fig_exp7_map.png'); plt.show()

In [ ]:
# ===== EXP7-c: 高복원율 ext 근처 정밀 랜덤 + s1 복원 + 맵 =====
from tqdm.notebook import trange
try: log
except NameError: log = []
LEAK_TH = 0.99
cand = [r for r in log if len(r)>=6 and r[4] in ('LEAK','faulty') and r[5] not in ('',None)]
cand.sort(key=lambda r: float(r[5]), reverse=True)
base_e = int(cand[0][0]) if cand else int(WIN*0.5)
print('정밀화 중심 ext =', base_e, '| 후보 %d개' % len(cand), flush=True)
cnt = collections.OrderedDict(golden=0, LEAK=0, faulty=0, mute=0); best=None; best_agr=0.0
for _ in trange(120, desc='refine'):
    e = max(base_e + random.randint(-800,800), 0)
    w = random.randint(35,65); o = random.randint(-12,12); rep = random.choice([1,2])
    set_glitch(e, w, o); scope.glitch.repeat = int(rep)
    g, p = glitch_once(); agr=None
    if g=='mute': cls='mute'
    elif g=='normal': cls='golden'
    else:
        agr=recover_agreement(); best_agr=max(best_agr,agr)
        if agr>=LEAK_TH:
            cls='LEAK'; print('★ LEAK s1 %.1f%% (ext=%d w=%d o=%d rep=%d)'%(100*agr,e,w,o,rep))
            if best is None: best=(int(e),int(w),int(o),int(rep))
        else: cls='faulty'
    cnt[cls]+=1
    log.append((int(e),int(w),int(o),int(rep),cls,('%.3f'%agr) if agr is not None else '', p.hex() if p else ''))
scope.glitch.repeat=1
print('done:', dict(cnt), '| best s1=%.1f%%'%(100*best_agr), '| best LEAK:', best)
with open('exp7_glitch_log.csv','w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['ext','width','offset','repeat','class','agreement','digest']); wr.writerows(log)
import matplotlib.pyplot as plt
pts=[(r[0],r[1],r[4]) for r in log if len(r)>=5 and r[4] in ('LEAK','faulty')]
fig,ax=plt.subplots(figsize=(8,3.4),dpi=130)
for e,w,c_ in pts:
    if c_=='LEAK': ax.scatter(e,w,c='red',marker='+',s=120,linewidths=2)
    else: ax.scatter(e,w,facecolors='none',edgecolors='gray',s=25)
ax.set_xlabel('ext_offset'); ax.set_ylabel('width'); ax.set_title('HAETAE HW glitch map (red+ = LEAK)')
fig.tight_layout(); fig.savefig('fig_exp7_map.png'); plt.show()

In [ ]:
# ===== EXP7-d: 축 A 변형 비교 (baseline/double/leeha/irv, 복원율 기반) + 막대그래프 =====
from tqdm.notebook import trange
assert TRIG_POINT == FL['ADDY'], 'TRIG_POINT=FL[ADDY]로 EXP7-a 실행 필요'
N_PER    = 60
W_RANGE  = (35,70); O_RANGE=(-15,15); REP_POOL=[1,1,2]
E_MAX    = int(WIN*1.05) if WIN>0 else 4000
LEAK_TH  = 0.99
VARIANTS = ['baseline','double','leeha','irv']
CLASSES  = ['golden','LEAK','faulty','mute']
CCOL     = {'golden':'0.8','LEAK':'red','faulty':'orange','mute':'black'}
summary = collections.OrderedDict(); glog=[]
for V in VARIANTS:
    scope.io.hs2='clkgen'
    flash('haetae-{}-FSIM-{}.hex'.format(V, PLATFORM))
    set_fault(FL['NONE'],0,0)
    scope.io.hs2='glitch'; scope.adc.timeout=3
    time.sleep(0.2); reset_target(scope); time.sleep(0.5); target.flush(); ss_trig(TRIG_POINT)
    if ss_sign() is None: recover_target(); ss_sign()
    cnt=collections.OrderedDict(golden=0,LEAK=0,faulty=0,mute=0); best_agr=0.0
    bar=trange(N_PER, desc='[%s]'%V)
    for i in bar:
        w=random.randint(*W_RANGE); o=random.randint(*O_RANGE); e=random.randint(0,E_MAX); rep=random.choice(REP_POOL)
        set_glitch(e,w,o); scope.glitch.repeat=int(rep)
        g,p=glitch_once(); agr=None
        if g=='mute': cls='mute'
        elif g=='normal': cls='golden'
        else:
            agr=recover_agreement(); best_agr=max(best_agr,agr)
            cls='LEAK' if agr>=LEAK_TH else 'faulty'
            if cls=='LEAK': bar.write('  [%s] LEAK s1 %.1f%% (ext=%d w=%d)'%(V,100*agr,e,w))
        cnt[cls]+=1; glog.append((V,e,w,o,rep,cls,('%.3f'%agr) if agr is not None else ''))
        bar.set_postfix(best_s1='%.0f%%'%(100*best_agr), **cnt)
    scope.glitch.repeat=1; summary[V]=dict(cnt); summary[V]['best_s1']=best_agr
    print('[%s] %s | best s1=%.1f%%'%(V,dict(cnt),100*best_agr), flush=True)

print('\n===== 축 A HW 클럭글리치 @+y — 변형 비교 (LEAK=s1복원>=%.0f%%) ====='%(100*LEAK_TH))
print('%-9s %6s %8s %8s %6s %8s'%('variant','LEAK','golden','faulty','mute','best_s1'))
for V,s in summary.items():
    print('%-9s %6d %8d %8d %6d %7.0f%%'%(V,s['LEAK'],s['golden'],s['faulty'],s['mute'],100*s['best_s1']))
print('\n해석: baseline은 LEAK(s1 복원) 발생, double/leeha/irv는 LEAK=0(복원율 낮음)이면 축 A로도 대응기법 실효 입증.')

import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4)); x=range(len(VARIANTS)); bottom=[0]*len(VARIANTS)
for cls in CLASSES:
    vals=[summary[V][cls] for V in VARIANTS]
    ax.bar(x,vals,bottom=bottom,color=CCOL[cls],label=cls,edgecolor='white')
    for xi,(vv,bb) in enumerate(zip(vals,bottom)):
        if vv: ax.text(xi,bb+vv/2,str(vv),ha='center',va='center',color=('white' if cls in ('LEAK','mute') else 'black'),fontsize=8)
    bottom=[b+v for b,v in zip(bottom,vals)]
ax.set_xticks(list(x)); ax.set_xticklabels(VARIANTS); ax.set_ylabel('count (N=%d)'%N_PER)
ax.legend(loc='upper right',fontsize=8); ax.set_title('Axis-A HW glitch @+y: baseline leaks, countermeasures block')
fig.tight_layout(); fig.savefig('fig_exp7_variants.png',dpi=130); fig.savefig('fig_exp7_variants.pdf'); plt.show()
with open('exp7_variants.csv','w',newline='') as f:
    wr=csv.writer(f); wr.writerow(['variant','LEAK','golden','faulty','mute','best_s1'])
    for V,s in summary.items(): wr.writerow([V,s['LEAK'],s['golden'],s['faulty'],s['mute'],'%.3f'%s['best_s1']])
    wr.writerow([]); wr.writerow(['variant','ext','width','offset','repeat','class','agreement']); wr.writerows(glog)
print('saved exp7_variants.csv + fig_exp7_variants.{png,pdf}')

## 해석 / 운영

- **판정**: `LEAK`=결함 응답에서 s1 복원율 ≥ `LEAK_TH`(기본 0.99). `faulty`=결함이나 복원 낮음. `golden`=무영향. `mute`=무응답(크래시).
- **진행바 `best_s1`** 를 보세요. 높으면 실제 누설, 낮게 머물면 단일 글리치가 +y를 부분 교란만 하는 것.
- LEAK이 안 나오면: `LEAK_TH` 낮춰 부분누설 관찰 / `EXP7-a`의 `TRIG_POINT=FL['CS']`(T1) 로 바꿔 재실행 / width·ext 범위 조정.
- **운영 팁**: mute 폭주 시 타겟이 꼬입니다. **커널 재시작 → EXP7-a → (즉시) EXP7-b2/b** 가 항상 깨끗한 출발점.
- 산출물: `exp7_glitch_log.csv`, `exp7_variants.csv`, `fig_exp7_map.png`, `fig_exp7_variants.{png,pdf}`.

**주의**: `T` 트리거 명령이 있는 최신 펌웨어로 빌드된 `haetae-*-FSIM-*.hex` 필요(없으면 메인 노트북 BUILD 셀 또는 WSL make 로 재빌드).